# Module 4 - Class 1: Regression Modeling - Sales Forecasting
**Khamidullokhon Abduvokhidov**

Upload `Sample - Superstore.csv` to Colab before running.

In [ ]:
# Load and inspect Superstore data, then check the predictor relationships.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')
features = ['Quantity', 'Discount', 'Profit']
print(df.shape)
display(df[features + ['Sales']].describe())
print(df[features].isnull().sum())
df = df.dropna(subset=features + ['Sales'])
corr = df[['Sales'] + features].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Feature Correlations with Sales')
plt.show()

In [ ]:
# Fit and evaluate the baseline linear regression model.
X, y = df[features], df['Sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)
def metrics(actual, predicted):
    mse = mean_squared_error(actual, predicted)
    return [mse, np.sqrt(mse), mean_absolute_error(actual, predicted), r2_score(actual, predicted)]
linear_metrics = metrics(y_test, y_pred)
print(pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_}))
print('Intercept:', model.intercept_)
print('A one-unit change in a feature changes predicted sales by its coefficient while the other features stay fixed.')

In [ ]:
# Add degree-two terms and compare polynomial regression with the baseline.
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
model_poly = LinearRegression().fit(X_train_poly, y_train)
y_pred_poly = model_poly.predict(X_test_poly)
comparison = pd.DataFrame([linear_metrics, metrics(y_test, y_pred_poly)], index=['Linear Regression', 'Polynomial Regression'], columns=['MSE', 'RMSE', 'MAE', 'R2'])
display(comparison)
print('Polynomial features help only if their test metrics improve; otherwise their added complexity is not justified.')

In [ ]:
# Use predicted-versus-actual and residual plots to diagnose the linear model.
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5, s=20)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Sales'); plt.ylabel('Predicted Sales'); plt.title('Predicted vs Actual Sales'); plt.show()
residuals = y_test - y_pred
plt.figure(figsize=(8, 6))
plt.scatter(y_pred, residuals, alpha=0.5, s=20)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Sales'); plt.ylabel('Residuals'); plt.title('Residual Plot'); plt.show()